In [1]:
import pandas as pd
import requests

NASDAQ_URL = "ftp://ftp.nasdaqtrader.com/SymbolDirectory/nasdaqlisted.txt"
OTHER_URL = "ftp://ftp.nasdaqtrader.com/SymbolDirectory/otherlisted.txt"

def fetch_nasdaq_list():
    df = pd.read_csv(NASDAQ_URL, sep="|")
    return df

def fetch_other_listed():
    df = pd.read_csv(OTHER_URL, sep="|")
    return df

nasdaq_df = fetch_nasdaq_list()
other_df = fetch_other_listed()

print("NASDAQ count:", len(nasdaq_df))
print("Other Exchanges count:", len(other_df))

NASDAQ count: 5319
Other Exchanges count: 6984


In [2]:
nasdaq_df.to_csv("data/nasdaq.csv", index=False)
other_df.to_csv("data/other.csv", index=False)

In [6]:
import yfinance as yf
import pandas as pd
from tqdm import tqdm


# =====================================================
# 1️⃣ Build Clean Master Ticker List
# =====================================================

def clean_string_columns(df):
    """
    Strip whitespace only for string/object columns.
    Compatible with pandas 2.x.
    """
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.strip()
    return df


def get_exchange_tickers(
    nasdaq_path="data/nasdaq.csv",
    other_path="data/other.csv"
):

    # Read as string to avoid float/NaN issues
    nasdaq = pd.read_csv(nasdaq_path, dtype=str).fillna("")
    other = pd.read_csv(other_path, dtype=str).fillna("")

    nasdaq = clean_string_columns(nasdaq)
    other = clean_string_columns(other)

    # ----------------------------
    # Filter NASDAQ
    # ----------------------------
    nasdaq = nasdaq[
        (nasdaq["ETF"] == "N") &
        (nasdaq["Test Issue"] == "N")
    ]

    nasdaq_tickers = (
        nasdaq["Symbol"]
        .astype(str)
        .str.strip()
        .replace("", pd.NA)
        .dropna()
        .tolist()
    )

    # ----------------------------
    # Filter Other Exchanges
    # ----------------------------
    other = other[
        (other["ETF"] == "N") &
        (other["Test Issue"] == "N")
    ]

    other_tickers = (
        other["ACT Symbol"]
        .astype(str)
        .str.strip()
        .replace("", pd.NA)
        .dropna()
        .tolist()
    )

    # Combine
    combined = nasdaq_tickers + other_tickers

    # Remove weird Yahoo-breaking tickers (optional but recommended)
    combined = [t for t in combined if "." not in t]

    # Deduplicate safely
    tickers = sorted(set(combined))

    print(f"NASDAQ cleaned count: {len(nasdaq_tickers)}")
    print(f"Other cleaned count: {len(other_tickers)}")
    print(f"Final unique tickers: {len(tickers)}")

    return tickers


# =====================================================
# 2️⃣ Chunk Helper
# =====================================================

def chunk_list(lst, chunk_size):
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]


# =====================================================
# 3️⃣ Batch Downloader
# =====================================================

def download_price_data(
    tickers,
    start="2005-01-01",
    chunk_size=200
):

    all_data = []

    for batch in tqdm(list(chunk_list(tickers, chunk_size))):

        try:
            df = yf.download(
                batch,
                start=start,
                group_by="ticker",
                auto_adjust=False,
                progress=False,
                threads=True
            )

            if df.empty:
                continue

            if isinstance(df.columns, pd.MultiIndex):

                for ticker in batch:
                    if ticker not in df.columns.levels[0]:
                        continue

                    temp = df[ticker].copy().reset_index()

                    temp.columns = [
                        col.lower().replace(" ", "_")
                        for col in temp.columns
                    ]

                    # Enforce numeric types
                    numeric_cols = ["open", "high", "low", "close", "adj_close", "volume"]
                    for col in numeric_cols:
                        if col in temp.columns:
                            temp[col] = pd.to_numeric(temp[col], errors="coerce")

                    temp["ticker"] = ticker

                    all_data.append(temp)

            else:
                df = df.reset_index()
                df.columns = [
                    col.lower().replace(" ", "_")
                    for col in df.columns
                ]

                numeric_cols = ["open", "high", "low", "close", "adj_close", "volume"]
                for col in numeric_cols:
                    if col in df.columns:
                        df[col] = pd.to_numeric(df[col], errors="coerce")

                df["ticker"] = batch[0]
                all_data.append(df)

        except Exception:
            continue

    if not all_data:
        print("No data downloaded.")
        return

    final = pd.concat(all_data, ignore_index=True)
    final = final.sort_values(["ticker", "date"])

    final.to_parquet("data/prices.parquet")

    print("Saved prices to data/prices.parquet")


# =====================================================
# 4️⃣ Run
# =====================================================

tickers = get_exchange_tickers()
download_price_data(tickers)


NASDAQ cleaned count: 4176
Other cleaned count: 3137
Final unique tickers: 7191


  0%|          | 0/36 [00:00<?, ?it/s]HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AGM$H"}}}
$AGM$H: possibly delisted; no timezone found
$AGBK: possibly delisted; no timezone found
$ADC$A: possibly delisted; no timezone found
$AGM$G: possibly delisted; no timezone found
$ABR$D: possibly delisted; no timezone found
Failed to get ticker 'AEAQ' reason: Failed to perform, curl: (28) Operation timed out after 10002 milliseconds with 186 out of 1182 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$AEAQ: possibly delisted; no timezone found
$AGM$D: possibly delisted; no timezone found
$ACR$C: possibly delisted; no timezone found
$ACR$D: possibly delisted; no timezone found
$ACP$A: possibly delisted; no timezone found
$AGM$F: possibly delisted; no timezone found
$AGM$E: possibly delisted; no timezone found
$ABR$E: possibly delisted; no timezone found
$ABR$F: possibly delisted; no

Saved prices to data/prices.parquet


In [27]:
prices = pd.read_parquet('data/prices.parquet')
prices

,date,open,high,low,close,adj_close,volume,ticker
0,2005-01-03,17.238913,17.296137,16.809729,17.081545,14.273739,3378826.0,A
1,2005-01-04,17.010014,17.160229,16.487841,16.630901,13.897172,3746920.0,A
2,2005-01-05,16.573677,16.917025,16.537910,16.623749,13.891194,3898603.0,A
3,2005-01-06,16.738197,16.766809,16.230330,16.258942,13.586355,3158641.0,A
4,2005-01-07,16.223175,16.416309,16.187410,16.244635,13.574401,2624326.0,A
...,...,...,...,...,...,...,...,...
29814595,2026-02-04,NaN,NaN,NaN,NaN,NaN,NaN,UGP
29814596,2026-02-05,NaN,NaN,NaN,NaN,NaN,NaN,UGP
29814597,2026-02-06,NaN,NaN,NaN,NaN,NaN,NaN,UGP
29814598,2026-02-09,NaN,NaN,NaN,NaN,NaN,NaN,UGP


In [28]:
prices['ticker'].nunique()

5800

In [1]:
from universe import UniverseBuilder
builder = UniverseBuilder()

dates = builder.get_month_end_dates()

print("First rebalance date:", dates[0])
print("First 5 dates:", dates[:5])

u = builder.get_universe(dates[0])
print("Universe size:", len(u))


First rebalance date: 2005-04-29 00:00:00
First 5 dates: [Timestamp('2005-04-29 00:00:00'), Timestamp('2005-05-31 00:00:00'), Timestamp('2005-06-30 00:00:00'), Timestamp('2005-07-29 00:00:00'), Timestamp('2005-08-31 00:00:00')]
Universe size: 991


In [36]:
dates = builder.get_month_end_dates()

print("First 10 rebalance dates:")
print(dates[:10])

First 10 rebalance dates:
[Timestamp('2005-01-31 00:00:00'), Timestamp('2005-02-28 00:00:00'), Timestamp('2005-03-31 00:00:00'), Timestamp('2005-04-29 00:00:00'), Timestamp('2005-05-31 00:00:00'), Timestamp('2005-06-30 00:00:00'), Timestamp('2005-07-29 00:00:00'), Timestamp('2005-08-31 00:00:00'), Timestamp('2005-09-30 00:00:00'), Timestamp('2005-10-31 00:00:00')]


In [38]:
dates = builder.get_month_end_dates()
print(dates[:5])


[Timestamp('2005-01-31 00:00:00'), Timestamp('2005-02-28 00:00:00'), Timestamp('2005-03-31 00:00:00'), Timestamp('2005-04-29 00:00:00'), Timestamp('2005-05-31 00:00:00')]
